# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR<sup>2</sup>](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, following the [Croissant](https://mlcommons.org/croissant/) schema standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Title**: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- **Authors**: Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M.
- **Description**: This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes fields such as accession, description, coverage percentage, peptide counts, molecular weight, pI, and normalized abundances across different samples.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant JSON-LD schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}\n\nVersion: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available **record sets**, **fields**, and their `@id`s.

We enumerate the available record sets and their field definitions using the Croissant metadata. For each record set, you can retrieve its `@id`, `name`, and associated fields (`@id` and `name`).

In [ ]:
# List available record sets and their fields by @id
from pprint import pprint

record_sets = list(metadata.record_set)
print(f"Number of record sets: {len(record_sets)}\n")

record_set_overview = {}
for rs in record_sets:
    print(f"Record set: @id={rs.id}, name={rs.name}")
    fields = list(rs.field)
    field_infos = []
    for field in fields:
        # Some fields may not have a name
        field_infos.append({'@id': field.id, 'name': getattr(field, 'name', None)})
    record_set_overview[rs.id] = field_infos
    for field in field_infos:
        print(f"    Field: @id={field['@id']}, name={field['name']}")
    print()

# For further use, pick one main record set (first in the list):
if record_sets:
    main_record_set_id = record_sets[0].id
    main_record_set_fields = [f['@id'] for f in record_set_overview[main_record_set_id]]


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We use the record set and field `@id`s discovered above.

For demonstration, we extract data from **all** record sets into DataFrames and inspect the columns present in one main set.

In [ ]:
# Extract data from each record set using their @id
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set '@id': {record_set_id}")

if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in the main record set (@id={main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
  - Filtering records based on specific criteria (e.g., numeric field threshold)
  - Normalizing numeric fields
  - Grouping by attributes for analysis

**All fields are referenced by their `@id`.**

> 💡 To adapt for your analysis, swap out the `numeric_field_id` and `groupby_field_id` as needed.

In [ ]:
# Example: Filter, normalize, and group using @id fields
# Replace these @id strings with real field IDs found in the overview step above
main_df = dataframes[main_record_set_id]

# Choose likely numeric field and group field by @id (adjust as needed based on printed field lists above)
# Example field candidates: 'cr:protein_coverage', 'cr:peptide_count', 'cr:MW', 'cr:normalized_abundance', etc.
numeric_field_id = None
groupby_field_id = None

for col in main_df.columns:
    # Heuristically select 'coverage' or 'peptide_count' field as numeric
    if 'coverage' in col.lower() or 'peptide_count' in col.lower() or 'mw' in col.lower():
        numeric_field_id = col
        break

# Heuristically choose a grouping field (e.g. 'cr:sample' or 'cr:description')
for col in main_df.columns:
    if 'sample' in col.lower() or 'type' in col.lower() or 'description' in col.lower():
        groupby_field_id = col
        break

if numeric_field_id is not None:
    threshold = 10
    # Force conversion to numeric (some data may be string)
    filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize numeric column
    filtered_df[numeric_field_id + '_normalized'] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by group field if exists
    if groupby_field_id is not None and groupby_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(groupby_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {groupby_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field automatically detected. Adjust code for your field IDs.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following example displays a histogram of the selected numeric field, referenced by its `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and not main_df[numeric_field_id].isnull().all():
    plt.figure(figsize=(7, 4))
    sns.histplot(pd.to_numeric(main_df[numeric_field_id], errors='coerce').dropna(), bins=20, kde=True)
    plt.xlabel(f"{numeric_field_id}")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.tight_layout()
    plt.show()
else:
    print("Numeric field not set or is empty. Adjust your selection above.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR<sup>2</sup> dataset using `mlcroissant`. We examined its metadata, record sets, and fields (all referenced by `@id`), extracted data, performed basic filtering and normalization, grouped by key attributes, and visualized distributions. For rich analysis, reference the dataset browser and Croissant docs for full field definitions and metadata context.

- **All data elements are referenced by their Croissant `@id` fields** for clarity and reproducibility.
- Customize field selection (`@id`) in code cells above to match your research question.

For more, see:
- [FAIR<sup>2</sup> Dataset on SenScience](https://sen.science/doi/10.71728/senscience.vq4a-28xa)
- [`mlcroissant` documentation](https://mlcommons.org/croissant/)
